# Stock Performance and Fundamental Analysis

**Author:** Medwin Odamtten  
**Tools:** Python, pandas, NumPy, yfinance, Matplotlib and Seaborn

## Project question

How have five well-known stocks performed relative to the S&P 500, and what can their historical returns, risk measures and company fundamentals tell us about the tradeoffs between growth and risk?

## Companies in the starting portfolio

- Apple (`AAPL`)
- Microsoft (`MSFT`)
- Nvidia (`NVDA`)
- Amazon (`AMZN`)
- JPMorgan Chase (`JPM`)
- S&P 500 benchmark (`SPY`)

You can change these symbols in the configuration cell. The analysis uses public market data and is for education only. It is **not investment advice**, and historical performance does not guarantee future results.

### How to run it in Google Colab

Upload this notebook, open the **Runtime** menu, and select **Run all**. Run the cells from top to bottom whenever you change a stock symbol or date.

### What this notebook demonstrates

1. Downloading and checking market data.
2. Calculating total return, annualized return, volatility, maximum drawdown and Sharpe ratio.
3. Comparing stocks with a market benchmark.
4. Examining diversification with a correlation heatmap.
5. Collecting selected company fundamentals.
6. Studying price behavior around earnings announcements.
7. Reviewing recent financial-news headlines without pretending that headlines prove causation.


## 1. Install the data package

Google Colab already includes most data-analysis libraries. The following command installs `yfinance`, which provides convenient access to publicly available Yahoo Finance market data.


In [ ]:
# Install yfinance quietly so Colab does not print a long installation log.
%pip -q install yfinance

## 2. Import the libraries

Each library has a different job: pandas manages tables, NumPy performs numerical calculations, yfinance retrieves market data, and Matplotlib/Seaborn create charts.


In [ ]:
# Import warnings so we can hide distracting non-critical messages.
import warnings

# Import NumPy for mathematical operations such as square roots.
import numpy as np

# Import pandas for tables, dates, cleaning and grouped calculations.
import pandas as pd

# Import Matplotlib for creating and customizing charts.
import matplotlib.pyplot as plt

# Import Seaborn for a clean correlation heatmap.
import seaborn as sns

# Import yfinance for historical prices, fundamentals, earnings dates and news.
import yfinance as yf

# Import display so tables render neatly inside a notebook.
from IPython.display import display

# Hide non-critical warnings to keep the notebook output readable.
warnings.filterwarnings("ignore")

# Use a clean chart style that makes gridlines easy to read.
sns.set_theme(style="whitegrid")

# Show two decimal places in ordinary pandas tables.
pd.options.display.float_format = "{:,.2f}".format

# Confirm that the setup cell ran successfully.
print("Libraries loaded successfully.")

## 3. Choose the stocks and assumptions

This is the main configuration cell. Edit it when you want to analyze different companies, a different time period or a different assumed risk-free rate.


In [ ]:
# List the five individual stocks we want to study.
STOCKS = ["AAPL", "MSFT", "NVDA", "AMZN", "JPM"]

# Choose SPY as a tradable benchmark for the S&P 500.
BENCHMARK = "SPY"

# Combine the stocks and benchmark into one download list.
ALL_TICKERS = STOCKS + [BENCHMARK]

# Start with January 2021 to create several years of comparable history.
START_DATE = "2021-01-01"

# Use None so yfinance downloads through the latest available trading day.
END_DATE = None

# Assume 252 trading days in an average year.
TRADING_DAYS = 252

# Use a configurable 4% annual risk-free rate for the simplified Sharpe ratio.
RISK_FREE_RATE = 0.04

# Choose one company for the detailed drawdown, earnings and news sections.
FOCUS_TICKER = "AAPL"

# Print the configuration so the reader can confirm what will be analyzed.
print("Stocks:", STOCKS)

# Print the benchmark symbol separately for clarity.
print("Benchmark:", BENCHMARK)

# Print the analysis start date.
print("Start date:", START_DATE)

## 4. Download adjusted closing prices

Adjusted prices account for events such as stock splits and cash distributions. Using adjusted data makes long-run return comparisons more meaningful than using unadjusted closing prices.


In [ ]:
# Download historical market data for every selected symbol in one request.
raw_data = yf.download(
    ALL_TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
)

# Select the adjusted Close field from yfinance's multi-level result.
prices = raw_data["Close"].copy()

# Reorder the columns so they match the list defined above.
prices = prices.reindex(columns=ALL_TICKERS)

# Remove dates where every selected symbol is missing.
prices = prices.dropna(how="all")

# Fill an occasional missing observation using the most recent available price.
prices = prices.ffill()

# Stop with a clear message if no data was downloaded.
if prices.empty:
    raise ValueError("No price data was downloaded. Check the ticker symbols or internet connection.")

# Display the first five rows so we can inspect the table structure.
display(prices.head())

# Print the number of trading days and symbols in the cleaned table.
print(f"Cleaned dataset: {prices.shape[0]:,} trading days and {prices.shape[1]} symbols.")

### Data-quality check

Before calculating returns, we check missing values and the available date range. This avoids treating a partial data series as if it covered the entire period.


In [ ]:
# Count the remaining missing observations in each price column.
missing_values = prices.isna().sum().rename("Missing values")

# Find the first valid date for every symbol.
first_dates = prices.apply(lambda column: column.first_valid_index()).rename("First valid date")

# Find the final valid date for every symbol.
last_dates = prices.apply(lambda column: column.last_valid_index()).rename("Last valid date")

# Combine the three checks into one readable table.
quality_check = pd.concat([missing_values, first_dates, last_dates], axis=1)

# Display the quality-check table.
display(quality_check)

## 5. Compare the growth of a hypothetical $100

Stocks have very different share prices, so plotting the raw prices can be misleading. Rebasing every series to $100 shows the percentage growth of an equal starting investment.


In [ ]:
# Divide every price by its first available price and multiply by 100.
growth_of_100 = prices.divide(prices.iloc[0]).multiply(100)

# Create a wide figure so the full history is easy to read.
ax = growth_of_100.plot(figsize=(14, 7), linewidth=2)

# Add a title describing exactly what the chart compares.
ax.set_title("Growth of a Hypothetical $100 Investment", fontsize=16, weight="bold")

# Label the horizontal axis.
ax.set_xlabel("Date")

# Label the vertical axis in dollar terms.
ax.set_ylabel("Value of Starting $100")

# Draw a reference line at the original $100 investment.
ax.axhline(100, color="black", linewidth=1, linestyle="--", alpha=0.6)

# Move the legend outside the plotting area so it does not cover the lines.
ax.legend(title="Symbol", bbox_to_anchor=(1.02, 1), loc="upper left")

# Adjust spacing so labels and the external legend are not cut off.
plt.tight_layout()

# Display the completed chart.
plt.show()

## 6. Calculate returns and risk measures

### Metric definitions

- **Total return:** Overall percentage change from the first price to the final price.
- **Annualized return:** Compound yearly rate implied by the complete period.
- **Annualized volatility:** Standard deviation of daily returns scaled to one year. Higher volatility means wider price movement, not automatically a worse investment.
- **Maximum drawdown:** Largest percentage decline from a previous high.
- **Sharpe ratio:** Simplified estimate of excess annual return per unit of annual volatility. It depends on the chosen risk-free rate and should not be treated as a complete investment decision.


In [ ]:
# Convert consecutive prices into daily percentage returns.
daily_returns = prices.pct_change(fill_method=None).dropna(how="all")

# Create a function so the same calculations are applied consistently to every symbol.
def calculate_metrics(price_series, return_series):
    # Remove missing prices before using the first and last observations.
    clean_prices = price_series.dropna()

    # Remove missing returns before calculating averages and volatility.
    clean_returns = return_series.dropna()

    # Calculate the total percentage change across the complete period.
    total_return = clean_prices.iloc[-1] / clean_prices.iloc[0] - 1

    # Count the number of observed daily returns.
    observation_count = len(clean_returns)

    # Convert the total return into a compound annual growth estimate.
    annualized_return = (1 + total_return) ** (TRADING_DAYS / observation_count) - 1

    # Scale daily standard deviation by the square root of 252 trading days.
    annualized_volatility = clean_returns.std() * np.sqrt(TRADING_DAYS)

    # Track the highest price reached up to each date.
    running_peak = clean_prices.cummax()

    # Express every price as a percentage above or below its earlier peak.
    drawdown = clean_prices / running_peak - 1

    # Select the most negative drawdown as the maximum historical decline.
    maximum_drawdown = drawdown.min()

    # Calculate a simplified annual Sharpe ratio when volatility is nonzero.
    sharpe_ratio = (
        (annualized_return - RISK_FREE_RATE) / annualized_volatility
        if annualized_volatility > 0
        else np.nan
    )

    # Return all five measures with clear column names.
    return {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Maximum Drawdown": maximum_drawdown,
        "Sharpe Ratio": sharpe_ratio,
    }

# Calculate the metric dictionary for every symbol.
metric_records = {
    symbol: calculate_metrics(prices[symbol], daily_returns[symbol])
    for symbol in ALL_TICKERS
}

# Convert the nested dictionaries into a table with symbols as rows.
metrics = pd.DataFrame(metric_records).T

# Sort the table from highest to lowest annualized return.
metrics = metrics.sort_values("Annualized Return", ascending=False)

# Create a separate display copy so percentages appear in a reader-friendly format.
metrics_display = metrics.copy()

# Format the return and risk columns as percentages.
for column in ["Total Return", "Annualized Return", "Annualized Volatility", "Maximum Drawdown"]:
    # Apply percentage formatting without changing the numeric source table.
    metrics_display[column] = metrics_display[column].map(lambda value: f"{value:.2%}")

# Format the Sharpe ratio to two decimal places.
metrics_display["Sharpe Ratio"] = metrics_display["Sharpe Ratio"].map(lambda value: f"{value:.2f}")

# Display the completed comparison table.
display(metrics_display)

### Risk-versus-return chart

The upper-left region is generally attractive because it represents higher historical return with lower historical volatility. However, a chart of past data cannot establish which stock will perform best in the future.


In [ ]:
# Create a chart area for the risk-versus-return comparison.
fig, ax = plt.subplots(figsize=(10, 7))

# Plot annualized volatility on the x-axis and annualized return on the y-axis.
ax.scatter(
    metrics["Annualized Volatility"],
    metrics["Annualized Return"],
    s=130,
    color="#168A8A",
    alpha=0.85,
)

# Add a readable symbol label beside every point.
for symbol, row in metrics.iterrows():
    # Position the label slightly away from the point.
    ax.annotate(
        symbol,
        (row["Annualized Volatility"], row["Annualized Return"]),
        xytext=(6, 6),
        textcoords="offset points",
        weight="bold",
    )

# Label the chart with its analytical purpose.
ax.set_title("Historical Risk Versus Return", fontsize=16, weight="bold")

# Label volatility as the horizontal measure.
ax.set_xlabel("Annualized Volatility")

# Label return as the vertical measure.
ax.set_ylabel("Annualized Return")

# Format both axes as percentages.
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

# Format the return axis as percentages too.
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

# Use tight layout to prevent labels from being clipped.
plt.tight_layout()

# Display the completed scatter plot.
plt.show()

## 7. Examine diversification with correlation

Correlation ranges from -1 to +1. Values close to +1 mean two investments often move in the same direction. Lower correlation can improve diversification, although relationships can change during market stress.


In [ ]:
# Calculate the pairwise correlation of daily returns.
correlation_matrix = daily_returns.corr()

# Create a square figure for the heatmap.
plt.figure(figsize=(9, 7))

# Plot correlation values with a centered red-to-blue color scale.
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)

# Add a descriptive title.
plt.title("Correlation of Daily Returns", fontsize=16, weight="bold")

# Adjust spacing around the heatmap.
plt.tight_layout()

# Display the completed heatmap.
plt.show()

## 8. Study drawdowns for one company

A drawdown measures the decline from the highest price previously reached. It helps show how uncomfortable an investment could have felt even if its long-run return eventually became positive.


In [ ]:
# Select the focus company's cleaned price history.
focus_prices = prices[FOCUS_TICKER].dropna()

# Calculate the highest price reached up to each date.
focus_peak = focus_prices.cummax()

# Calculate the percentage distance below the previous peak.
focus_drawdown = focus_prices / focus_peak - 1

# Create a figure for the drawdown history.
fig, ax = plt.subplots(figsize=(14, 5))

# Plot the drawdown line.
ax.plot(focus_drawdown.index, focus_drawdown, color="#B04A5A", linewidth=1.8)

# Shade the area between zero and the negative drawdown values.
ax.fill_between(focus_drawdown.index, focus_drawdown, 0, color="#B04A5A", alpha=0.25)

# Add the selected company symbol to the title.
ax.set_title(f"{FOCUS_TICKER} Drawdown From Previous High", fontsize=16, weight="bold")

# Label the horizontal axis.
ax.set_xlabel("Date")

# Label and format the vertical axis as a percentage.
ax.set_ylabel("Drawdown")

# Display percentage labels on the vertical axis.
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

# Adjust chart spacing.
plt.tight_layout()

# Display the completed drawdown chart.
plt.show()

# Print the worst drawdown in a sentence that is easy to interpret.
print(f"Worst {FOCUS_TICKER} drawdown in this period: {focus_drawdown.min():.2%}")

## 9. Compare selected company fundamentals

Fundamental data provides another perspective beyond price. The values below can change over time and may occasionally be unavailable from the data source. Always verify important figures against company filings before making an investment decision.


In [ ]:
# Define the yfinance fields and the readable labels we want in the final table.
FUNDAMENTAL_FIELDS = {
    "marketCap": "Market Cap",
    "trailingPE": "Trailing P/E",
    "revenueGrowth": "Revenue Growth",
    "earningsGrowth": "Earnings Growth",
    "profitMargins": "Profit Margin",
    "debtToEquity": "Debt to Equity",
}

# Create an empty list that will store one row for every company.
fundamental_rows = []

# Loop through only the individual stocks, excluding the SPY benchmark.
for symbol in STOCKS:
    # Begin each row with the stock symbol.
    row = {"Symbol": symbol}

    # Use a protected block because web-based fundamentals can occasionally be unavailable.
    try:
        # Request the latest company information from yfinance.
        company_info = yf.Ticker(symbol).info

        # Copy each selected field into the row using its readable label.
        for source_name, display_name in FUNDAMENTAL_FIELDS.items():
            # Use get so a missing field becomes None instead of stopping the notebook.
            row[display_name] = company_info.get(source_name)
    except Exception as error:
        # Fill unavailable fields with missing values if the request fails.
        for display_name in FUNDAMENTAL_FIELDS.values():
            # Store NumPy NaN so pandas recognizes the value as missing.
            row[display_name] = np.nan

        # Print a short explanation instead of a long technical error message.
        print(f"Fundamentals were unavailable for {symbol}: {error}")

    # Add the completed company row to the list.
    fundamental_rows.append(row)

# Convert the list of company dictionaries into a pandas table.
fundamentals = pd.DataFrame(fundamental_rows).set_index("Symbol")

# Convert debt-to-equity from Yahoo's percentage-like scale into a ratio when present.
fundamentals["Debt to Equity"] = fundamentals["Debt to Equity"] / 100

# Create a copy specifically for reader-friendly display formatting.
fundamentals_display = fundamentals.copy()

# Convert market capitalization to billions of dollars.
fundamentals_display["Market Cap"] = fundamentals_display["Market Cap"].map(
    lambda value: f"${value / 1_000_000_000:,.1f}B" if pd.notna(value) else "N/A"
)

# Format P/E and debt-to-equity as ordinary ratios.
for column in ["Trailing P/E", "Debt to Equity"]:
    # Preserve missing values as N/A and show available ratios to two decimal places.
    fundamentals_display[column] = fundamentals_display[column].map(
        lambda value: f"{value:.2f}" if pd.notna(value) else "N/A"
    )

# Format growth and margin fields as percentages.
for column in ["Revenue Growth", "Earnings Growth", "Profit Margin"]:
    # Preserve missing values as N/A and show available values as percentages.
    fundamentals_display[column] = fundamentals_display[column].map(
        lambda value: f"{value:.2%}" if pd.notna(value) else "N/A"
    )

# Display the completed fundamental comparison.
display(fundamentals_display)

### Visualize growth and valuation

The chart below compares recent revenue growth with the trailing price-to-earnings ratio. A high growth rate can help explain a higher valuation, but neither measure guarantees future performance.


In [ ]:
# Select the two columns needed for the growth-versus-valuation chart.
growth_valuation = fundamentals[["Revenue Growth", "Trailing P/E"]].dropna()

# Continue only when the data source returned at least one complete company row.
if not growth_valuation.empty:
    # Create a chart area for the comparison.
    fig, ax = plt.subplots(figsize=(10, 7))

    # Plot trailing P/E on the x-axis and revenue growth on the y-axis.
    ax.scatter(
        growth_valuation["Trailing P/E"],
        growth_valuation["Revenue Growth"],
        s=140,
        color="#17365D",
        alpha=0.85,
    )

    # Label every point with its stock symbol.
    for symbol, row in growth_valuation.iterrows():
        # Move each label slightly away from its point.
        ax.annotate(
            symbol,
            (row["Trailing P/E"], row["Revenue Growth"]),
            xytext=(6, 6),
            textcoords="offset points",
            weight="bold",
        )

    # Add a title that identifies both comparison measures.
    ax.set_title("Recent Revenue Growth Versus Trailing P/E", fontsize=16, weight="bold")

    # Label the valuation axis.
    ax.set_xlabel("Trailing Price-to-Earnings Ratio")

    # Label the growth axis.
    ax.set_ylabel("Recent Revenue Growth")

    # Format revenue growth as a percentage.
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

    # Adjust chart spacing.
    plt.tight_layout()

    # Display the completed comparison chart.
    plt.show()
else:
    # Explain why no chart appears when web fundamentals are unavailable.
    print("Growth and valuation data were unavailable, so this chart was skipped.")

## 10. Earnings-event analysis

This section examines returns around recent earnings announcements for the focus company. The calculation describes price behavior near an event; it does **not** prove that the announcement caused the price movement.


In [ ]:
# Request a small set of recent earnings dates for the focus company.
try:
    # Download up to eight recent reported earnings events.
    earnings_dates = yf.Ticker(FOCUS_TICKER).get_earnings_dates(limit=8).head(8)
except Exception as error:
    # Create an empty table if the web request is unavailable.
    earnings_dates = pd.DataFrame()

    # Explain the skipped section in plain language.
    print(f"Earnings dates were unavailable: {error}")

# Continue only when the data source returned event dates.
if earnings_dates is not None and not earnings_dates.empty:
    # Create an empty list to hold one result for each earnings date.
    event_rows = []

    # Loop through the reported earnings dates returned by yfinance.
    for event_timestamp in earnings_dates.index:
        # Remove timezone information so the event can be compared with the price index.
        event_date = pd.Timestamp(event_timestamp).tz_localize(None).normalize()

        # Find the first trading date on or after the announcement date.
        following_dates = focus_prices.index[focus_prices.index >= event_date]

        # Skip an event when there is no later price observation in our dataset.
        if len(following_dates) == 0:
            # Move directly to the next event.
            continue

        # Store the matching trading day.
        trading_date = following_dates[0]

        # Find the integer position of that trading day in the price series.
        position = focus_prices.index.get_loc(trading_date)

        # Continue only when one earlier day and five later trading days are available.
        if position >= 1 and position + 5 < len(focus_prices):
            # Calculate the return from the preceding close to the event-day close.
            event_day_return = focus_prices.iloc[position] / focus_prices.iloc[position - 1] - 1

            # Calculate the return from the preceding close through five trading days later.
            five_day_return = focus_prices.iloc[position + 5] / focus_prices.iloc[position - 1] - 1

            # Add the calculated event result to our list.
            event_rows.append(
                {
                    "Earnings Date": event_date.date(),
                    "Matched Trading Date": trading_date.date(),
                    "Event-Day Return": event_day_return,
                    "Five-Trading-Day Return": five_day_return,
                }
            )

    # Convert the calculated event records into a table.
    event_study = pd.DataFrame(event_rows)

    # Continue only when at least one event had enough surrounding price data.
    if not event_study.empty:
        # Sort events from newest to oldest.
        event_study = event_study.sort_values("Earnings Date", ascending=False)

        # Create a formatted display copy.
        event_study_display = event_study.copy()

        # Format both return columns as percentages.
        for column in ["Event-Day Return", "Five-Trading-Day Return"]:
            # Apply percentage formatting to each return value.
            event_study_display[column] = event_study_display[column].map(lambda value: f"{value:.2%}")

        # Display the completed earnings-event table.
        display(event_study_display)
    else:
        # Explain why no event rows could be calculated.
        print("No earnings events had enough surrounding price data for this period.")
else:
    # Explain that the section is optional when the provider returns no events.
    print("No earnings dates were returned; the rest of the project is still complete.")

## 11. Review recent financial-news headlines

This section displays recent headlines supplied by the data provider. Headlines are useful research prompts, but they should be checked against reliable reporting and company filings. We do not assign automatic sentiment because short headlines often lack enough context.


In [ ]:
# Use a protected request because the news feed may change or be unavailable.
try:
    # Request recent news items connected with the focus company.
    raw_news = yf.Ticker(FOCUS_TICKER).news

    # Create an empty list for clean headline records.
    news_rows = []

    # Examine only the first ten returned items to keep the table concise.
    for item in raw_news[:10]:
        # Support yfinance versions that place article fields inside a content dictionary.
        content = item.get("content", item)

        # Retrieve the headline title when available.
        title = content.get("title")

        # Retrieve the publishing organization when available.
        provider = content.get("provider", {})

        # Convert a provider dictionary into its display name.
        publisher = provider.get("displayName") if isinstance(provider, dict) else provider

        # Retrieve the publication timestamp in its original form.
        published = content.get("pubDate") or content.get("providerPublishTime")

        # Add only records that contain a real headline.
        if title:
            # Store the clean headline information.
            news_rows.append(
                {
                    "Published": published,
                    "Publisher": publisher,
                    "Headline": title,
                }
            )

    # Convert the cleaned records into a table.
    recent_news = pd.DataFrame(news_rows)

    # Display the table when at least one headline was returned.
    if not recent_news.empty:
        # Show the recent-news table without an unnecessary numeric index.
        display(recent_news)
    else:
        # Explain why no headline table appears.
        print("No recent headlines were returned by the data provider.")
except Exception as error:
    # Explain a failed news request without stopping the rest of the notebook.
    print(f"Recent news was unavailable: {error}")

## 12. Generate careful, data-based observations

The code below automatically identifies a few notable historical facts. The language deliberately says *during this period* because the results depend on the chosen dates.


In [ ]:
# Find the symbol with the highest annualized historical return.
highest_return_symbol = metrics["Annualized Return"].idxmax()

# Find the symbol with the lowest annualized historical volatility.
lowest_volatility_symbol = metrics["Annualized Volatility"].idxmin()

# Find the symbol with the deepest historical maximum drawdown.
deepest_drawdown_symbol = metrics["Maximum Drawdown"].idxmin()

# Exclude self-correlations by replacing the diagonal with missing values.
correlations_without_diagonal = correlation_matrix.where(
    ~np.eye(correlation_matrix.shape[0], dtype=bool)
)

# Stack the matrix so each remaining pair becomes one row.
correlation_pairs = correlations_without_diagonal.stack()

# Remove duplicate reversed pairs such as AAPL-MSFT and MSFT-AAPL.
unique_pairs = correlation_pairs[
    [first < second for first, second in correlation_pairs.index]
]

# Identify the pair with the strongest positive correlation.
most_correlated_pair = unique_pairs.idxmax()

# Store the corresponding correlation value.
most_correlated_value = unique_pairs.max()

# Print the highest-return observation with its historical percentage.
print(
    f"1. {highest_return_symbol} had the highest annualized return during this period "
    f"at {metrics.loc[highest_return_symbol, 'Annualized Return']:.2%}."
)

# Print the lowest-volatility observation with its historical percentage.
print(
    f"2. {lowest_volatility_symbol} had the lowest annualized volatility during this period "
    f"at {metrics.loc[lowest_volatility_symbol, 'Annualized Volatility']:.2%}."
)

# Print the deepest-drawdown observation with its historical percentage.
print(
    f"3. {deepest_drawdown_symbol} experienced the deepest maximum drawdown during this period "
    f"at {metrics.loc[deepest_drawdown_symbol, 'Maximum Drawdown']:.2%}."
)

# Print the strongest-correlation observation and name both symbols.
print(
    f"4. {most_correlated_pair[0]} and {most_correlated_pair[1]} had the strongest daily-return "
    f"correlation at {most_correlated_value:.2f}."
)

# Remind the reader not to turn descriptive history into a future prediction.
print("5. These observations describe the selected historical period and are not forecasts.")

## 13. Export the analysis tables

Saving the numeric results makes the project reproducible and allows the tables to be opened in Excel, Tableau or another reporting tool.


In [ ]:
# Save the numeric performance and risk metrics without display formatting.
metrics.to_csv("stock_summary_metrics.csv")

# Save the latest available fundamental fields for the five individual companies.
fundamentals.to_csv("stock_fundamentals.csv")

# Confirm the names of the two exported files.
print("Saved stock_summary_metrics.csv and stock_fundamentals.csv")

## 14. Interpretation prompts

After running the notebook, answer these questions in your own words:

1. Which company had the highest return, and what additional risk accompanied it?
2. Which company behaved most similarly to the S&P 500?
3. Which two holdings were most highly correlated?
4. Did the fastest-growing company also have the highest valuation?
5. What happened around the focus company’s recent earnings dates?
6. Which limitation would you address in a future version?

### Limitations

- The selected companies create a small, non-random sample.
- Historical returns cannot predict future returns.
- A constant 4% risk-free rate is a simplifying assumption.
- Fundamental fields come from a third-party source and may be delayed or missing.
- Correlations can change, especially during unusual market periods.
- Price movement around earnings or news does not prove that the event caused the movement.
- Taxes, transaction costs, inflation and portfolio rebalancing are not modeled.

### Possible extensions

- Allow a user to enter portfolio weights and calculate weighted portfolio performance.
- Compare the portfolio with different benchmarks.
- Retrieve fundamentals directly from audited regulatory filings.
- Backtest a clearly defined strategy without using future information.
- Add rolling volatility and rolling correlation to show how risk changes over time.
